In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_pry")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyectoyr1")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/customers.csv"

In [0]:
customers_schema = StructType(fields=[
                    StructField("CustomerID", IntegerType(), False),
                    StructField("FirstName", StringType(), True),
                    StructField("MiddleInitial", StringType(), True),
                    StructField("LastName", StringType(), True),
                    StructField("CityID", IntegerType(), True),
                    StructField("Address", StringType(), True),
                    StructField("ingestion_date", TimestampType(), True)
])

In [0]:
customers_df = spark.read \
            .option("header", True) \
            .schema(customers_schema) \
            .csv(ruta)

In [0]:
customers_final_df = customers_df.withColumn("ingestion_date", current_timestamp())

In [0]:
customers_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.customers")